# LoLip on a toy problem

This notebook runs the paper's pipeline end to end on a problem small enough to train in a few minutes:

1. **Pretrain a base flow policy**, conditioned on the observation, on demonstrations of several tasks, without telling it
   which task each demonstration is for. Once with plain flow matching (**Vanilla CFM**), once with the lower-Lipschitz
   regularizer (**LoLip**). Asked for a particular task, the base alone can only guess.
2. **Steer the frozen base with DSBC.** Map new demonstrations back through the flow to their noises, train a noise
   policy conditioned on the observation and the task, and roll it out through the unchanged base.

The toy: a scene is a ring of 8 targets in one of 4 layouts (the ring rotated by a different angle); the observation is
the layout. Task k is "reach target k". An action is a chunk of 8 steps of a 2-D action, so the noise is 16-dimensional.
The base is pretrained on 256 demonstrations; DSBC gets 10 new demonstrations per task.

Use a GPU runtime (in Colab: Runtime → Change runtime type → GPU). Training takes a few minutes.

In [ ]:
import math, torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

## 1. Tasks and demonstrations

In [ ]:
K = 8                 # tasks: reach target k
H = 8                 # steps per action chunk
D = 2 * H             # action (and noise) dimension
SIGMA = 0.08          # spread of a demonstration around its target
COLORS = plt.cm.tab10(np.arange(K))

def targets(phi):                                 # a scene: the ring of K targets rotated by phi
    ang = phi[:, None] + 2 * math.pi * torch.arange(K)[None] / K
    return torch.stack([1.5 * torch.cos(ang), 1.5 * torch.sin(ang)], -1)        # (n, K, 2)

def observe(phi):                                 # what the policy sees
    return torch.stack([torch.cos(phi), torch.sin(phi)], 1)

LAYOUTS = 4

def layout_angle(n, g=None):                      # one of 4 layouts: the ring rotated by a quarter of the target spacing
    return torch.randint(0, LAYOUTS, (n,), generator=g).float() * 2 * math.pi / (LAYOUTS * K)

def demos(n, g=None):
    phi = layout_angle(n, g)
    task = torch.randint(0, K, (n,), generator=g)
    chunk = targets(phi)[torch.arange(n), task].repeat(1, H) + SIGMA * torch.randn(n, D, generator=g)
    return phi, task, chunk

def reached(actions, phi, task):                  # the first step of the chunk lands on the asked target
    return (torch.norm(actions[:, :2].cpu() - targets(phi)[torch.arange(len(phi)), task], dim=1) < 3 * SIGMA).float()

g = torch.Generator().manual_seed(0)
base_phi, base_task, base_demos = demos(256, g)   # pretraining data; the base never sees the task
dsbc_phi, dsbc_task, dsbc_demos = demos(10 * K, g)   # new demonstrations, used only by DSBC

fig, ax = plt.subplots(1, LAYOUTS, figsize=(3.4 * LAYOUTS, 3.4))
for x, phi in zip(ax, torch.arange(LAYOUTS).float()[:, None] * 2 * math.pi / (LAYOUTS * K)):
    T_ = targets(phi)[0]
    x.scatter(T_[:, 0], T_[:, 1], s=120, c=COLORS)
    for k in range(K): x.annotate(str(k), T_[k], ha="center", va="center", fontsize=8, color="w")
    x.set_title(f"layout {round(phi.item() * LAYOUTS * K / (2 * math.pi))}"); x.set_aspect("equal"); x.set_xlim(-2, 2); x.set_ylim(-2, 2)
plt.tight_layout(); plt.show()

## 2. Flow policy, flow matching, and the LoLip regularizer

A flow policy turns noise `z ~ N(0, I)` into an action by integrating a learned velocity field `v(x, t)` over `N` Euler
steps from `t = 0` to `t = 1`. Flow matching trains `v` so that, on the straight line `x_t = (1 - t) z + t a` from a noise
to a demonstrated action, it predicts `a - z`. Running the same Euler steps backwards maps an action to its noise.

**LoLip** adds one term to the loss: for a random unit direction `u`, it penalises any Euler step that shrinks `u` below
`s_min = T^(-1/N)`, so the whole flow shrinks no direction by more than a factor `T`.

In [ ]:
class Flow(nn.Module):                            # v(x, t | condition)
    def __init__(self, dim, cond, h=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim + 1 + cond, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, dim))
    def forward(self, x, t, c):
        return self.net(torch.cat([x, t[:, None], c], 1))

N = 50                                            # Euler steps

def generate(v, z, c):                            # noise -> action, given the condition
    x = z
    for i in range(N):
        x = x + v(x, torch.full((len(x),), i / N, device=x.device), c) / N
    return x

def invert(v, a, c):                              # action -> noise (the same steps, backwards)
    x = a
    for i in reversed(range(N)):
        x = x - v(x, torch.full((len(x),), i / N, device=x.device), c) / N
    return x

def lolip_penalty(v, x_t, t, c, T):
    s_min = T ** (-1 / N)                                         # smallest allowed stretch per step
    u = torch.randn_like(x_t); u = u / u.norm(dim=1, keepdim=True) # random unit direction
    step = lambda x: x + v(x, t, c) / N                           # one Euler step
    _, Ju = torch.autograd.functional.jvp(step, x_t, u, create_graph=True)
    return 0.5 * torch.relu(s_min - Ju.norm(dim=1)).mean()        # penalise shrinking below s_min

def flow_matching(data, cond, lam=0.0, T=10.0, steps=15000, seed=0):
    torch.manual_seed(seed)
    data, cond = data.to(dev), cond.to(dev)
    v = Flow(data.shape[1], cond.shape[1]).to(dev)
    opt = torch.optim.Adam(v.parameters(), 2e-3)
    for _ in range(steps):
        i = torch.randint(0, len(data), (512,), device=dev)
        a, c = data[i], cond[i]
        z = torch.randn_like(a); t = torch.rand(len(a), device=dev)
        x_t = (1 - t[:, None]) * z + t[:, None] * a
        loss = ((v(x_t, t, c) - (a - z)) ** 2).sum(1).mean()      # flow matching
        if lam > 0:
            loss = loss + lam * lolip_penalty(v, x_t, t, c, T)    # + LoLip
        opt.zero_grad(); loss.backward(); opt.step()
    return v.eval()

## 3. Pretrain the two base policies

Both are conditioned on the observation only. Same data, same network, same training; LoLip only adds the penalty
(λ = 50, T = 10).

In [ ]:
vanilla = flow_matching(base_demos, observe(base_phi))
lolip = flow_matching(base_demos, observe(base_phi), lam=50.0, T=10.0)

## 4. DSBC on each base

For each base: map the new demonstrations back to their noises through the frozen base, train a noise policy on them
(a small flow-matching model conditioned on the task), then for each task sample a noise from the noise policy and run it
through the base. The base's weights never change.

In [ ]:
def dsbc(base, seed=0, n=2000):
    with torch.no_grad():
        w = invert(base, dsbc_demos.to(dev), observe(dsbc_phi).to(dev))       # the new demos' noises
    cond = torch.cat([observe(dsbc_phi), nn.functional.one_hot(dsbc_task, K).float()], 1)
    noise_policy = flow_matching(w, cond, steps=8000, seed=seed)              # noise policy: (observation, task) -> noise
    g = torch.Generator().manual_seed(1)
    phi = layout_angle(n, g); task = torch.randint(0, K, (n,), generator=g)   # scenes and asked tasks
    s = observe(phi).to(dev)
    with torch.no_grad():
        alone = generate(base, torch.randn(n, D, device=dev), s)              # the base alone, plain Gaussian noise
        z = generate(noise_policy, torch.randn(n, D, device=dev), torch.cat([s, nn.functional.one_hot(task, K).float().to(dev)], 1))
        steered = generate(base, z, s)                                        # DSBC: the noise policy's noise through the base
    return dict(w=w.cpu(), phi=phi, task=task, alone=alone.cpu(), steered=steered.cpu(),
                base_success=reached(alone, phi, task).mean().item(), dsbc_success=reached(steered, phi, task).mean().item())

results = {"Vanilla CFM": dsbc(vanilla), "LoLip": dsbc(lolip)}
print(f"chance: {1 / K:.0%}; expected norm of a 16-D Gaussian noise: {math.sqrt(D):.1f}")
for name, r in results.items():
    print(f"{name:12} base alone on the asked task: {r['base_success']:.0%}   with DSBC: {r['dsbc_success']:.0%}"
          f"   median norm of the new demos' noises: {r['w'].norm(dim=1).median():.1f}")

## 5. Results

Left: success on the asked task, for the base alone and with DSBC. Middle: the new demonstrations mapped back to noise
through each base, coloured by task: the first two of the 16 noise coordinates, on a window where a Gaussian sample would
almost always fall (dotted circle: 2 standard deviations). Right: DSBC
rollouts, drawn relative to the asked target (green: reached).

In [ ]:
def show(results):
    fig, ax = plt.subplots(len(results), 3, figsize=(12, 3.8 * len(results)), squeeze=False)
    for row, (name, r) in zip(ax, results.items()):
        row[0].bar(["base alone", "with DSBC"], [r["base_success"], r["dsbc_success"]], color=["#b4b4b8", "#d91e5b"])
        row[0].axhline(1 / K, ls=":", c="k", lw=1); row[0].text(1.45, 1 / K, "chance", va="bottom", ha="right", fontsize=8)
        row[0].set_ylim(0, 1); row[0].set_title(f"{name}: success on the asked task", fontsize=10)
        for i, v in enumerate([r["base_success"], r["dsbc_success"]]): row[0].text(i, v + .02, f"{v:.0%}", ha="center", fontsize=9)
        p = r["w"][:, :2]                                   # first two noise coordinates, on a Gaussian-scale window
        off = int((p.abs() > 4).any(1).sum())
        row[1].scatter(p[:, 0], p[:, 1], s=8, c=COLORS[dsbc_task]); row[1].add_patch(plt.Circle((0, 0), 2, fill=False, ls=":"))
        row[1].set_xlim(-4, 4); row[1].set_ylim(-4, 4); row[1].set_aspect("equal")
        row[1].set_title(f"new demonstrations mapped back to noise\n{off} of {len(p)} outside this window; median norm {r['w'].norm(dim=1).median():.1f}", fontsize=10)
        rel = r["steered"][:, :2] - targets(r["phi"])[torch.arange(len(r["phi"])), r["task"]]
        ok = reached(r["steered"], r["phi"], r["task"]).bool()
        row[2].scatter(rel[:, 0], rel[:, 1], s=2, c=np.where(ok[:, None], [[0.16, .62, .29, 1]], [[.6, .6, .6, 1]]))
        row[2].add_patch(plt.Circle((0, 0), 3 * SIGMA, fill=False, ls="--"))
        row[2].set_xlim(-3.2, 3.2); row[2].set_ylim(-3.2, 3.2); row[2].set_aspect("equal")
        row[2].set_title(f"DSBC rollouts, relative to the asked target\nreached {r['dsbc_success']:.0%}", fontsize=10)
    plt.tight_layout(); plt.show()

show(results)

**Reading the plots.** The base alone does not know which target was asked for, so it lands on the asked one about once
in 8 tries. DSBC should fix that by learning, per observation and task, which noise to give the base. With the Vanilla CFM
base it does not: Vanilla contracts large regions of noise onto each tight target, so its reverse map is strongly
expansive, and new demonstrations map to noises far outside the Gaussian (median norm well above √16 = 4). A noise
policy cannot learn those noises, and DSBC does no better than the base alone. LoLip bounds the contraction, the same
demonstrations map to noises of Gaussian scale, the noise policy learns them, and DSBC reaches the asked target many
times more often than the base alone.

## 6. Play with the regularizer

Move λ (the weight of the penalty) and T (the largest contraction allowed over the whole flow), then press **Train**.
λ = 0 is Vanilla CFM. Each run pretrains a new base and runs DSBC on it (about a minute on a GPU).

In [ ]:
def try_lolip(lam, T):
    base = flow_matching(base_demos, observe(base_phi), lam=lam, T=T)
    show({"Vanilla CFM": results["Vanilla CFM"], f"LoLip (λ = {lam:g}, T = {T:g})": dsbc(base)})

try:
    import ipywidgets as W
    from IPython.display import display, clear_output
    lam_w = W.FloatLogSlider(value=50, base=10, min=-1, max=2.5, description="λ")
    T_w = W.FloatLogSlider(value=10, base=10, min=0.3, max=2, description="T")
    button = W.Button(description="Train", button_style="primary"); out = W.Output()
    def on_click(_):
        with out:
            clear_output(wait=True); print("training ...")
            lam, T = lam_w.value, T_w.value
            clear_output(wait=True); try_lolip(lam, T)
    button.on_click(on_click)
    display(W.HBox([lam_w, T_w, button]), out)
except ImportError:
    print("ipywidgets is not installed; call try_lolip(lam, T) directly, for example:")
    try_lolip(5.0, 10.0)

**Things to try**

* Small λ (0.5, 5): how much regularization does DSBC need before it works?
* T = 2 (a strict bound) versus T = 100 (a loose one) at λ = 50.

The paper applies the same regularizer to flow policies for robot manipulation (Isaac Lab, LIBERO, a real robot), where
the policy is also conditioned on the observation.